In [5]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(url, names=columns)

print("First 5 rows:")
print(df.head())

print("\nDataset shape:", df.shape)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[zero_columns] = X[zero_columns].replace(0, np.nan)
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()


class LogisticRegressionGD:

    def __init__(self, learning_rate=0.01, iterations=10000):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.iterations):
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)

            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict_probability(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)

    def predict(self, X):
        probabilities = self.predict_probability(X)
        return (probabilities >= 0.5).astype(int)


print("\n" + "=" * 60)
print("LOGISTIC REGRESSION WITHOUT FEATURE SCALING")
print("=" * 60)

model_without_scaling = LogisticRegressionGD(
    learning_rate=0.00001,
    iterations=20000
)

model_without_scaling.fit(X_train, y_train)

y_pred_without = model_without_scaling.predict(X_test)


print("\n" + "=" * 60)
print("LOGISTIC REGRESSION WITH FEATURE SCALING")
print("=" * 60)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_with_scaling = LogisticRegressionGD(
    learning_rate=0.01,
    iterations=10000
)

model_with_scaling.fit(X_train_scaled, y_train)

y_pred_with = model_with_scaling.predict(X_test_scaled)


def evaluate_model(y_true, y_pred, model_name):

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("\n" + model_name)
    print("-" * 40)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1-score :", round(f1, 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["No Diabetes", "Diabetes"],
            zero_division=0
        )
    )

    return accuracy, precision, recall, f1


results_without = evaluate_model(
    y_test,
    y_pred_without,
    "Without Feature Scaling"
)

results_with = evaluate_model(
    y_test,
    y_pred_with,
    "With Feature Scaling"
)

comparison = pd.DataFrame(
    {
        "Without Scaling": results_without,
        "With Scaling": results_with
    },
    index=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]
)

print("\n" + "=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)

print(comparison.round(4))



First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset shape: (768, 9)

LOGISTIC REGRESSION WITHOUT FEATURE SCALING

LOGISTIC REGRESSION WITH FEATURE SCALING

Without Feature Scaling
----------------------------------------
Accuracy : 0.7078
Precision: 0.6452
Recall   : 0.3704
F1-score : 0.4706

Confusion Matrix:
[[89 11]
 [34 20]]

